In [ ]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_example
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,37,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch315_valAcc0.9571_valPrec0.6716_valRecall0.7050.h5')#
#cnn_model.load_weights('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training_subset/filtered_poly_data.tfrecord'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_example).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

Image height:  148
Before string split: (None, 148, 64), max_x=148.0


Model: "guitar_note_detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_spectrogram   │ (None, 148, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_mean          │ (None, 148, 256,  │          0 │ input_spectrogra… │
│ (AveragePooling2D)  │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_contrast      │ (None, 148, 256,  │          0 │ input_spectrogra… │
│ (Subtract)          │ 1)                │            │ local_mean[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_2d       │ (None, 148, 256,  │          0 │ local_contrast[0… │
│ (Reshape)           │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_conv… │ (None, 148, 64,   │        136 │ reshape_to_2d[0]… │
│ (Conv2D)            │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_bn    │ (None, 148, 64,   │         32 │ freq_compress_co… │
│ (BatchNormalizatio… │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_act   │ (None, 148, 64,   │          0 │ freq_compress_bn… │
│ (LeakyReLU)         │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_drop  │ (None, 148, 64,   │          0 │ freq_compress_ac… │
│ (SpatialDropout2D)  │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_1d       │ (None, 148, 512)  │          0 │ freq_compress_dr… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_squeeze    │ (None, 148, 32)   │     16,416 │ reshape_to_1d[0]… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_squeeze_bn │ (None, 148, 32)   │        128 │ backbone_squeeze… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_squeeze_a… │ (None, 148, 32)   │          0 │ backbone_squeeze… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_conv1      │ (None, 148, 32)   │      7,200 │ backbone_squeeze… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_bn1        │ (None, 148, 32)   │        128 │ backbone_conv1[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_act1       │ (None, 148, 32)   │          0 │ backbone_bn1[0][… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_drop1      │ (None, 148, 32)   │          0 │ backbone_act1[0]… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ backbone_conv2      │ (None, 148, 64)   │     14,400 │ backbone_drop1[0

 Total params: 503,309 (1.92 MB)

 Trainable params: 499,837 (1.91 MB)

...

INFO:tensorflow:Assets written to: /tmp/tmpz6pg4n2d/assets


INFO:tensorflow:Assets written to: /tmp/tmpz6pg4n2d/assets


Saved artifact at '/tmp/tmpz6pg4n2d'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 148, 256, 1), dtype=tf.float32, name='input_spectrogram')
Output Type:
  TensorSpec(shape=(None, 37), dtype=tf.float32, name=None)
Captures:
  139857174439184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139857174440144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139857174438800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139857174437840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139857174439568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139857174439376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139857174440336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139857174441488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139857174440528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139857174439952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13985717

W0000 00:00:1775220982.697545 1207638 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1775220982.697563 1207638 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1775220982.697787 1207638 reader.cc:83] Reading SavedModel from: /tmp/tmpz6pg4n2d
I0000 00:00:1775220982.700034 1207638 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1775220982.700043 1207638 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpz6pg4n2d
I0000 00:00:1775220982.726199 1207638 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1775220982.732930 1207638 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1775220982.920515 1207638 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpz6pg4n2d
I0000 00:00:1775220982.969730 1207638 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 271953 microseconds.
I0000 00:00:1775220983.026050 120763

: 